# Data Exploration for SNLI-VE

In [1]:
import json
import pandas as pd
import pyarrow as pa
import os

from tqdm import tqdm
from collections import defaultdict

In [2]:
root = '/home/claytonfields/nlp/code/data/SNLI-VE/data'
dataset_root = '.'

In [3]:
train_data = list(
    map(json.loads, open(f"{root}/snli_ve_train.jsonl").readlines())
)
test_data = list(
    map(json.loads, open(f"{root}/snli_ve_test.jsonl").readlines())
)
dev_data = list(
    map(json.loads, open(f"{root}/snli_ve_dev.jsonl").readlines())
)

In [4]:
train_data

[{'Flickr30K_ID': '4564320256',
  'annotator_labels': ['contradiction'],
  'captionID': '4564320256.jpg#1',
  'gold_label': 'contradiction',
  'pairID': '4564320256.jpg#1r1c',
  'sentence1': 'An old lady and her granddaughter working in a convenience store.',
  'sentence1_binary_parse': '( ( ( ( ( An ( old lady ) ) and ) ( her granddaughter ) ) ( working ( in ( a ( convenience store ) ) ) ) ) . )',
  'sentence1_parse': '(ROOT (NP (NP (NP (DT An) (JJ old) (NN lady)) (CC and) (NP (PRP$ her) (NN granddaughter))) (VP (VBG working) (PP (IN in) (NP (DT a) (NN convenience) (NN store)))) (. .)))',
  'sentence2': 'Two old men robbing a convenience store.',
  'sentence2_binary_parse': '( ( ( Two ( old men ) ) ( robbing ( a ( convenience store ) ) ) ) . )',
  'sentence2_parse': '(ROOT (NP (NP (CD Two) (JJ old) (NNS men)) (VP (VBG robbing) (NP (DT a) (NN convenience) (NN store))) (. .)))'},
 {'Flickr30K_ID': '4564320256',
  'annotator_labels': ['entailment'],
  'captionID': '4564320256.jpg#1',
  '

In [5]:
label2id = {'contradiction': 0, 'neutral': 1, 'entailment': 2}
def process(root, imgid, ann):
    with open(f"{root}/Flickr30K/images/{imgid}.jpg", "rb") as fp:
        img = fp.read()

    sentences = ann['sentences']

    labels = ann['labels']

    return [img, sentences, labels]





train_data = list(
    map(json.loads, open(f"{root}/snli_ve_train.jsonl").readlines())
)
test_data = list(
    map(json.loads, open(f"{root}/snli_ve_test.jsonl").readlines())
)
dev_data = list(
    map(json.loads, open(f"{root}/snli_ve_dev.jsonl").readlines())
)


split = 'dev'


annotations = dict()
annotations['train'] = train_data
annotations['dev'] = dev_data
annotations['test'] = test_data
annots = dict()

annots[split] = {}
for line in annotations[split]:
    imgid = line['Flickr30K_ID']
    if not imgid in annots[split]:
        annots[split][imgid] = {}
        annots[split][imgid]['sentences'] = []
        annots[split][imgid]['labels'] = []
    annots[split][imgid]['sentences'].append( [line['sentence1'], line['sentence2']] )
    annots[split][imgid]['labels'].append( label2id[line['gold_label']] )
    
    

bs = [process(root, imgid, annots[split][imgid]) for imgid in tqdm(annots[split])]

dataframe = pd.DataFrame(
    bs, columns=["image", "sentences", "labels"]
)

table = pa.Table.from_pandas(dataframe)

# os.makedirs(dataset_root, exist_ok=True)
# with pa.OSFile(f"{dataset_root}/snli_{split}.arrow", "wb") as sink:
#     with pa.RecordBatchFileWriter(sink, table.schema) as writer:
#         writer.write_table(table)

100%|████████████████████████████████████| 1000/1000 [00:00<00:00, 32201.46it/s]


In [6]:
dataframe

,image,sentences,labels
0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A toddler poses in front of a computer at a ...,"[0, 1, 2, 2, 1, 1, 1, 0, 2, 1, 0, 2, 0, 2, 1]"
1,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A woman sits on a bench next to a pay phone ...,"[2, 0, 1, 1, 2, 0, 0, 1, 2, 2, 1, 0, 0, 1, 2]"
2,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[Two boys pose in front of a prehistoric gard...,"[2, 1, 1, 0, 0, 0, 0, 1, 0, 2, 1, 1, 2, 2, 2, ..."
3,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[The man in the background is grilling food w...,"[0, 0, 1, 2, 2, 2, 1, 2, 0, 1, 1, 2, 0, 0, 1, ..."
4,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[Two guys playing football., Two guys playing...","[0, 1, 2, 2, 1, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0]"
...,...,...,...
995,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A young woman dressed in black and another w...,"[1, 0, 2, 1, 2, 0, 1, 0, 2, 0, 2, 1, 0, 1, 2]"
996,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[Four runners pose for a picture., Four men p...","[1, 0, 2, 0, 2, 1, 1, 2, 0, 1, 0, 2, 1, 0, 2]"
997,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[Several kids practicing the same karate move...,"[0, 1, 2, 1, 2, 0, 0, 2, 1, 1, 2, 0, 1, 2, 0]"
998,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[People crossing a city street., Men walking ...","[2, 1, 0, 1, 0, 2, 1, 2, 0, 0, 1, 2, 0, 2, 1]"


In [22]:
dataframe['sentences'][0]

[['A toddler poses in front of a computer at a business office.',
  'A toddler sleeps outside.'],
 ['A toddler poses in front of a computer at a business office.',
  'A toddler poses at the office.'],
 ['A toddler poses in front of a computer at a business office.',
  'A toddler poses in front of a computer indoors.'],
 ['An infant is sitting in front of a computer.',
  'The baby is in front of the computer.'],
 ['An infant is sitting in front of a computer.',
  'The baby loves the computer.'],
 ['An infant is sitting in front of a computer.',
  'The computer smells like diapers.'],
 ['A little asian girl is sitting at a computer desk about to grab the mouse.',
  'The little asian girl at the computer is about to play a game.'],
 ['A little asian girl is sitting at a computer desk about to grab the mouse.',
  'The little hispanic girl sits in front of the computer desk.'],
 ['A little asian girl is sitting at a computer desk about to grab the mouse.',
  'The asian girl sits at the comp

In [71]:
table.num_rows

1000

In [19]:
table.column_names

['image', 'sentences', 'labels']

## SNLI-VE Dataset from METER

In [38]:
from meter.datasets.snli_dataset import SNLIDataset
from meter.datasets.base_dataset import BaseDataset
from transformers import AutoTokenizer

In [14]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
data_dir = '/home/claytonfields/nlp/code/meter/data/arrow'
transform_keys = ['imagenet']
image_size = 224

dset = SNLIDataset(data_dir,
        transform_keys,
        image_size, 
        split="val",
        tokenizer=tokenizer
)
dset.tokenizer = tokenizer

In [24]:
index = 0
dset[index]

{'image': [tensor([[[ 0.7591,  0.7933,  0.7762,  ...,  0.6392,  0.5364,  0.5022],
           [ 0.7762,  0.7248,  0.7419,  ...,  0.6221,  0.4508,  0.4679],
           [ 0.7419,  0.7933,  0.7933,  ...,  0.6221,  0.6563,  0.5878],
           ...,
           [-2.0152, -1.9124, -1.7240,  ..., -1.9809, -2.0152, -2.0152],
           [-1.6898, -1.7069, -1.7583,  ..., -1.9809, -2.0152, -1.9809],
           [-1.4672, -1.5528, -1.5870,  ..., -1.9467, -1.9295, -1.9295]],
  
          [[ 0.6429,  0.6779,  0.6954,  ...,  0.7654,  0.8179,  0.7829],
           [ 0.7304,  0.7654,  0.7654,  ...,  0.8354,  0.8179,  0.8004],
           [ 0.7479,  0.7479,  0.7304,  ...,  0.7829,  0.8004,  0.7654],
           ...,
           [-1.9132, -1.7906, -1.6506,  ..., -1.8957, -1.9307, -1.9132],
           [-1.5980, -1.5280, -1.6155,  ..., -1.8957, -1.8957, -1.8782],
           [-1.4055, -1.4405, -1.4405,  ..., -1.8782, -1.8606, -1.8606]],
  
          [[ 0.5485,  0.4962,  0.5485,  ...,  0.6879,  0.5485,  0.6008],
  

In [28]:
image_tensor = dset.get_image(index)["image"]
image_tensor

[tensor([[[ 0.7591,  0.7933,  0.7762,  ...,  0.6392,  0.5364,  0.5022],
          [ 0.7762,  0.7248,  0.7419,  ...,  0.6221,  0.4508,  0.4679],
          [ 0.7419,  0.7933,  0.7933,  ...,  0.6221,  0.6563,  0.5878],
          ...,
          [-2.0152, -1.9124, -1.7240,  ..., -1.9809, -2.0152, -2.0152],
          [-1.6898, -1.7069, -1.7583,  ..., -1.9809, -2.0152, -1.9809],
          [-1.4672, -1.5528, -1.5870,  ..., -1.9467, -1.9295, -1.9295]],
 
         [[ 0.6429,  0.6779,  0.6954,  ...,  0.7654,  0.8179,  0.7829],
          [ 0.7304,  0.7654,  0.7654,  ...,  0.8354,  0.8179,  0.8004],
          [ 0.7479,  0.7479,  0.7304,  ...,  0.7829,  0.8004,  0.7654],
          ...,
          [-1.9132, -1.7906, -1.6506,  ..., -1.8957, -1.9307, -1.9132],
          [-1.5980, -1.5280, -1.6155,  ..., -1.8957, -1.8957, -1.8782],
          [-1.4055, -1.4405, -1.4405,  ..., -1.8782, -1.8606, -1.8606]],
 
         [[ 0.5485,  0.4962,  0.5485,  ...,  0.6879,  0.5485,  0.6008],
          [ 0.3742,  0.5659,

In [30]:
text = dset.get_text(index)["text"]
text

('A toddler sleeps outside.',
 {'input_ids': [101, 1037, 6927, 3917, 25126, 2648, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})

In [32]:
index, question_index = dset.index_mapper[index]
index

0

In [35]:
labels = dset.table["labels"][index][question_index].as_py()
labels

0

In [37]:
dset.table.column_names

['image', 'sentences', 'labels']

## BaseDataset from METER

In [41]:
split = 'val'
if split == "val":
    names = ["snli_dev", "snli_test"]

base = BaseDataset(data_dir,
        transform_keys,
        image_size, 
        names=names,
        text_column_name="sentences",
)
base.tokenizer = tokenizer

/tmp/ipykernel_42760/1609298963.py:5: FutureWarning: promote has been superseded by mode='default'.
  base = BaseDataset(data_dir,


In [45]:
base.get_text(0)

{'text': ('A toddler sleeps outside.',
  {'input_ids': [101, 1037, 6927, 3917, 25126, 2648, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}),
 'img_index': 0,
 'cap_index': 0,
 'raw_index': 0}

In [46]:
base.table.column_names

['image', 'sentences', 'labels']

In [55]:
data_dir = '/home/claytonfields/nlp/code/meter/data/arrow'
transform_keys = ['imagenet']
image_size = 224
# names = []
names = ["snli_dev", "snli_test"]
text_column_name = ""
remove_duplicate=True
max_text_len=40
draw_false_image=0
draw_false_text=0
image_only=False
tokenizer=None

"""
data_dir : where dataset file *.arrow lives; existence should be guaranteed via DataModule.prepare_data
transform_keys : keys for generating augmented views of images
text_column_name : pyarrow table column name that has list of strings as elements
"""
assert len(transform_keys) >= 1

# transforms = keys_to_transforms(transform_keys, size=image_size)
# clip_transform = False
# for transform_key in transform_keys:
#     if 'clip' in transform_key:
#         clip_transform = True
#         break
text_column_name = text_column_name
names = names
max_text_len = max_text_len
draw_false_image = draw_false_image
draw_false_text = draw_false_text
image_only = image_only
data_dir = data_dir

if len(names) != 0:
    tables = [
        pa.ipc.RecordBatchFileReader(
            pa.memory_map(f"{data_dir}/{name}.arrow", "r")
        ).read_all()
        for name in names
        if os.path.isfile(f"{data_dir}/{name}.arrow")
    ]

    table_names = list()
    for i, name in enumerate(names):
        table_names += [name] * len(tables[i])

    table = pa.concat_tables(tables, promote=True)
    if text_column_name != "":
        text_column_name = text_column_name
        all_texts = table[text_column_name].to_pandas().tolist()
        if type(all_texts[0][0]) == str:
            all_texts = (
                [list(set(texts)) for texts in all_texts]
                if remove_duplicate
                else all_texts
            )
        else: #snli
            all_texts = (
                [[t[1].strip() for t in texts] for texts in all_texts]
            )
    else:
        all_texts = list()
else:
    all_texts = list()

index_mapper = dict()

if text_column_name != "" and not image_only:
    j = 0
    for i, texts in enumerate(all_texts):
        for _j in range(len(texts)):
            index_mapper[j] = (i, _j)
            j += 1
else:
    for i in range(len(table)):
        index_mapper[i] = (i, None)

/home/claytonfields/anaconda3/envs/meter-test/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3526: FutureWarning: promote has been superseded by mode='default'.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [62]:
texts

NameError: name 'texts' is not defined

In [60]:
len(table)

2000

In [56]:
names

['snli_dev', 'snli_test']

In [57]:
table.column_names

['image', 'sentences', 'labels']

In [61]:
tables

[pyarrow.Table
 image: binary
 sentences: list<item: list<item: string>>
   child 0, item: list<item: string>
       child 0, item: string
 labels: list<item: int64>
   child 0, item: int64
 ----
 image: [[...]]
 sentences: [[[["A toddler poses in front of a computer at a business office.","A toddler sleeps outside."],["A toddler poses in front of a computer at a business office.","A toddler poses at the office."],...,["A small child sits in front of a computer.","A child sits by a computer."],["A small child sits in front of a computer.","A small child sits in front of a laptop computer."]],[["A woman sits on a bench next to a pay phone and faces train tracks and a large building.","The woman is sitting outside."],["A woman sits on a bench next to a pay phone and faces train tracks and a large building.","The car driver is taking a right turn."],...,["A woman wearing a vertical striped top, sitting on a bench beside a telephone booth in a train station.","A woman waits for a train."],

In [58]:
dataframe

,image,sentences,labels
0,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A toddler poses in front of a computer at a ...,"[0, 1, 2, 2, 1, 1, 1, 0, 2, 1, 0, 2, 0, 2, 1]"
1,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A woman sits on a bench next to a pay phone ...,"[2, 0, 1, 1, 2, 0, 0, 1, 2, 2, 1, 0, 0, 1, 2]"
2,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[Two boys pose in front of a prehistoric gard...,"[2, 1, 1, 0, 0, 0, 0, 1, 0, 2, 1, 1, 2, 2, 2, ..."
3,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[The man in the background is grilling food w...,"[0, 0, 1, 2, 2, 2, 1, 2, 0, 1, 1, 2, 0, 0, 1, ..."
4,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[Two guys playing football., Two guys playing...","[0, 1, 2, 2, 1, 0, 1, 2, 0, 1, 2, 0, 1, 2, 0]"
...,...,...,...
995,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[A young woman dressed in black and another w...,"[1, 0, 2, 1, 2, 0, 1, 0, 2, 0, 2, 1, 0, 1, 2]"
996,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[Four runners pose for a picture., Four men p...","[1, 0, 2, 0, 2, 1, 1, 2, 0, 1, 0, 2, 1, 0, 2]"
997,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,[[Several kids practicing the same karate move...,"[0, 1, 2, 1, 2, 0, 0, 2, 1, 1, 2, 0, 1, 2, 0]"
998,b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x01...,"[[People crossing a city street., Men walking ...","[2, 1, 0, 1, 0, 2, 1, 2, 0, 0, 1, 2, 0, 2, 1]"


In [52]:
all_texts

[]

In [53]:
tables

NameError: name 'tables' is not defined